# 2026 NBA Playoffs — Bracket Prediction


## Purpose
Apply trained models to the 2026 NBA playoff bracket. Monte Carlo simulation for champion probabilities.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys; sys.path.insert(0, str(Path(".").parent / "src"))
from nba_predictor.config import cfg
from nba_predictor.predict.output_formatter import format_bracket_markdown


## Load 2026 Predictions

In [ ]:
pred_dir = cfg.project_root / "data/predictions/2026"
series_path = pred_dir / "bracket_output.csv"
champ_path = pred_dir / "champion_probabilities.csv"
if series_path.exists():
    series_df = pd.read_csv(series_path)
    champ_df = pd.read_csv(champ_path)
    print(series_df.to_string())
else:
    print("Run make predict first")


## Series Probability Breakdown

In [ ]:
Path("../reports/figures").mkdir(parents=True, exist_ok=True)

ROUND_ORDER = ["first_round", "second_round", "conf_finals", "nba_finals"]
ROUND_LABELS = {"first_round": "R1", "second_round": "R2",
                "conf_finals": "CF", "nba_finals": "Finals"}

fig, axes = plt.subplots(1, 2, figsize=(15, 7))

# Left: win probability per series
ax = axes[0]
labels, win_probs, colors_bar = [], [], []
for rnd in ROUND_ORDER:
    for _, row in series_df[series_df["round"] == rnd].iterrows():
        label = f"{ROUND_LABELS[rnd]}: {row['higher_seed']} vs {row['lower_seed']}"
        labels.append(label)
        win_probs.append(row["p_higher_seed_wins"])
        colors_bar.append("steelblue" if row["p_higher_seed_wins"] >= 0.5 else "tomato")

y = np.arange(len(labels))
bars = ax.barh(y, [p * 100 for p in win_probs], color=colors_bar, alpha=0.8)
ax.axvline(50, color="black", lw=1, ls="--")
ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel("P(higher seed wins) %")
ax.set_title("2026 Predicted Win Probabilities by Series")
for bar, p in zip(bars, win_probs):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f"{p*100:.1f}%", va="center", fontsize=8)
ax.set_xlim(0, 75); ax.invert_yaxis(); ax.grid(alpha=0.3, axis="x")

# Right: stacked series length probabilities
ax2 = axes[1]
length_labels = []
p4 = series_df["p_length_4"].values * 100
p5 = series_df["p_length_5"].values * 100
p6 = series_df["p_length_6"].values * 100
p7 = series_df["p_length_7"].values * 100
for _, row in series_df.iterrows():
    rnd_label = ROUND_LABELS.get(row["round"], row["round"])
    length_labels.append(f"{rnd_label}: {row['higher_seed']} vs {row['lower_seed']}")

y2 = np.arange(len(length_labels))
ax2.barh(y2, p4, color="#2196F3", alpha=0.85, label="4 games")
ax2.barh(y2, p5, left=p4, color="#4CAF50", alpha=0.85, label="5 games")
ax2.barh(y2, p6, left=p4+p5, color="#FF9800", alpha=0.85, label="6 games")
ax2.barh(y2, p7, left=p4+p5+p6, color="#F44336", alpha=0.85, label="7 games")
ax2.set_yticks(y2); ax2.set_yticklabels(length_labels, fontsize=9)
ax2.set_xlabel("Probability (%)")
ax2.set_title("Predicted Series Length Distribution")
ax2.legend(loc="lower right", fontsize=9)
ax2.invert_yaxis(); ax2.grid(alpha=0.3, axis="x")

# Annotate modal pick
for i, row in series_df.reset_index().iterrows():
    modal = int(row["modal_length"])
    ax2.text(101, i, f"→{modal}g", va="center", fontsize=8, color="black")

plt.tight_layout()
plt.savefig("../reports/figures/09_series_probabilities.png", dpi=120, bbox_inches="tight")
plt.show()


## Championship Probability Distribution

In [ ]:
champ_sorted = champ_df.sort_values("p_champion", ascending=False).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Championship probability bar chart
ax = axes[0]
colors_champ = []
east = {"DET","BOS","NYK","CLE","MIA","PHI","ATL","ORL","TOR","MIL","IND","CHI","BRK","WAS","CHA","NYK"}
for _, row in champ_sorted.iterrows():
    colors_champ.append("steelblue" if row["team"] in east else "tomato")

bars = ax.bar(champ_sorted["team"], champ_sorted["p_champion"] * 100,
              color=colors_champ, alpha=0.85, edgecolor="white")
ax.set_xlabel("Team"); ax.set_ylabel("P(Champion) %")
ax.set_title("2026 NBA Championship Probabilities\n(blue=East, red=West)")
ax.tick_params(axis="x", rotation=45)
for bar, p in zip(bars, champ_sorted["p_champion"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f"{p*100:.1f}%", ha="center", va="bottom", fontsize=7)
ax.grid(alpha=0.3, axis="y")

# Round-by-round advancement
ax2 = axes[1]
top8 = champ_sorted.head(8)
rounds = ["p_r2", "p_conf_finals", "p_finals", "p_champion"]
round_labels = ["R2", "Conf Finals", "Finals", "Champion"]
x = np.arange(len(top8))
w = 0.2
colors_rnd = ["#90CAF9", "#42A5F5", "#1565C0", "gold"]
for i, (col, label, color) in enumerate(zip(rounds, round_labels, colors_rnd)):
    if col in top8.columns:
        ax2.bar(x + i*w, top8[col]*100, w, label=label, color=color, alpha=0.9)
ax2.set_xticks(x + 1.5*w)
ax2.set_xticklabels(top8["team"], rotation=45)
ax2.set_ylabel("Probability %")
ax2.set_title("Top-8 Teams: Round Advancement Probabilities")
ax2.legend(fontsize=9); ax2.grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("../reports/figures/09_championship_probabilities.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"\nPredicted champion: OKC ({champ_sorted.iloc[0]['p_champion']*100:.1f}%)")
print(f"Predicted Finals:   DET vs OKC")
print(f"\nTop 5 title contenders:")
for _, row in champ_sorted.head(5).iterrows():
    print(f"  {row['team']:4s}  {row['p_champion']*100:5.1f}%")


## Rendered Bracket

In [ ]:
print(format_bracket_markdown(2026))
